# 🥈 Silver Delta - Ecommerce Pedidos

## Squad 2 — Ecommerce

### Tabela: ecommerce_pedidos

## Objetivo

Realizar a limpeza, tratamento e validação dos dados da camada Bronze para Silver utilizando Delta Lake.

## Origem

Azure Data Lake Gen2:

`grupo3/bronze/ecommerce_pedidos`

## Destino

Azure Data Lake Gen2:

`grupo3/silver/ecommerce_pedidos`

## Processamento realizado

- Leitura incremental da camada Bronze;
- Remoção de duplicidades;
- Validação de campos obrigatórios;
- Padronização dos dados;
- Inclusão de `silver_processed_at`;
- Persistência Delta utilizando MERGE INTO.

In [0]:
# ==================================================
# IMPORTAÇÃO DAS BIBLIOTECAS
# ==================================================


# Funções Spark utilizadas para:
# - criação de colunas;
# - filtros;
# - validações;
# - tratamento de dados;
# - criação de timestamps.
from pyspark.sql import functions as F



# Funções específicas do Spark SQL utilizadas
# durante as regras de qualidade da camada Silver.
from pyspark.sql.functions import (
    col,
    trim,
    current_timestamp,
    when,
    lit
)



# Biblioteca utilizada para autenticação segura
# no Azure através do Service Principal.
from azure.identity import ClientSecretCredential



# Cliente utilizado para acessar o Azure Data Lake Gen2,
# permitindo validar caminhos e estruturas físicas.
from azure.storage.filedatalake import (
    DataLakeServiceClient
)



# Biblioteca Delta Lake responsável por operações
# avançadas como MERGE INTO (upsert).
#
# Será utilizada para atualizar a camada Silver
# sem gerar duplicidade.
from delta.tables import DeltaTable



print(
    "Bibliotecas Silver carregadas com sucesso!"
)

Bibliotecas Silver carregadas com sucesso!


In [0]:
# ==================================================
# CONFIGURAÇÃO AZURE DATA LAKE GEN2
# SILVER ECOMMERCE PEDIDOS
# ==================================================


# Secret Scope utilizado para buscar
# as credenciais de acesso ao Azure.
SECRET_SCOPE = "mercadata-ecommerce-pedidos"



# Storage Account do projeto.
storage_account = "internshipdatalake"



# Recupera credenciais armazenadas
# no Databricks Secret Scope.

client_id = dbutils.secrets.get(
    scope=SECRET_SCOPE,
    key="ADLS_CLIENT_ID"
)


tenant_id = dbutils.secrets.get(
    scope=SECRET_SCOPE,
    key="ADLS_TENANT_ID"
)


client_secret = dbutils.secrets.get(
    scope=SECRET_SCOPE,
    key="ADLS_CLIENT_SECRET"
)



# Configuração OAuth utilizada pelo Spark
# nas operações de leitura e gravação Delta.

adls_options = {

    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net":
        "OAuth",

    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",

    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net":
        client_id,

    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net":
        client_secret,

    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net":
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}



# ==================================================
# CAMINHOS DAS CAMADAS
# ==================================================


base_squad2 = (
    f"abfss://squad2@{storage_account}.dfs.core.windows.net"
)



# Origem: Bronze
caminho_bronze_pedidos = (
    f"{base_squad2}/grupo3/bronze/"
    "ecommerce_pedidos"
)



# Destino: Silver
caminho_silver_pedidos = (
    f"{base_squad2}/grupo3/silver/"
    "ecommerce_pedidos"
)



print(
    "Configuração Silver concluída!"
)


print(
    "\nBronze:"
)

print(
    caminho_bronze_pedidos
)


print(
    "\nSilver:"
)

print(
    caminho_silver_pedidos
)

Configuração Silver concluída!

Bronze:
abfss://squad2@internshipdatalake.dfs.core.windows.net/grupo3/bronze/ecommerce_pedidos

Silver:
abfss://squad2@internshipdatalake.dfs.core.windows.net/grupo3/silver/ecommerce_pedidos


In [0]:
# ==================================================
# LEITURA DA CAMADA BRONZE
# ECOMMERCE PEDIDOS
# ==================================================


# Leitura da tabela Bronze em formato Delta.
# A Silver sempre consome dados tratados tecnicamente
# pela camada anterior.

df_bronze = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(
        caminho_bronze_pedidos
    )
)



print(
    "Tabela Bronze carregada com sucesso!"
)



print(
    "\nQuantidade de registros:",
    df_bronze.count()
)



print(
    "\nSchema Bronze:"
)


df_bronze.printSchema()

Tabela Bronze carregada com sucesso!

Quantidade de registros: 774

Schema Bronze:
root
 |-- id_pedido: long (nullable = true)
 |-- id_cliente: long (nullable = true)
 |-- id_endereco_entrega: long (nullable = true)
 |-- dt_pedido: timestamp (nullable = true)
 |-- status_pedido: string (nullable = true)
 |-- valor_total: double (nullable = true)
 |-- valor_frete: double (nullable = true)
 |-- metodo_pagamento: string (nullable = true)
 |-- dt_previsao_entrega: timestamp (nullable = true)
 |-- dt_ultima_atualizacao_status: timestamp (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- bronze_source_file: string (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- dia: integer (nullable = true)
 |-- hora: integer (nullable = true)



In [0]:
# ==================================================
# LIMPEZA E VALIDAÇÃO SILVER
# ECOMMERCE PEDIDOS
# ==================================================


# Remover registros duplicados usando
# a chave de negócio do pedido.
df_silver = (
    df_bronze
    .dropDuplicates(
        ["id_pedido"]
    )
)



# Padronização de campos textuais
# removendo espaços desnecessários.
df_silver = (
    df_silver
    .withColumn(
        "status_pedido",
        F.trim(
            F.col("status_pedido")
        )
    )
    .withColumn(
        "metodo_pagamento",
        F.trim(
            F.col("metodo_pagamento")
        )
    )
)



# Filtrar registros válidos:
# - chaves obrigatórias preenchidas;
# - valores financeiros consistentes.

df_silver = (
    df_silver
    .filter(
        F.col("id_pedido").isNotNull()
    )
    .filter(
        F.col("id_cliente").isNotNull()
    )
    .filter(
        F.col("dt_pedido").isNotNull()
    )
    .filter(
        F.col("status_pedido").isNotNull()
    )
    .filter(
        F.col("valor_total") > 0
    )
    .filter(
        F.col("valor_frete") >= 0
    )
)



print(
    "Tratamento Silver realizado com sucesso!"
)


print(
    "Quantidade após tratamento:",
    df_silver.count()
)

Tratamento Silver realizado com sucesso!
Quantidade após tratamento: 766


In [0]:
# ==================================================
# ADIÇÃO DE METADADO SILVER
# ==================================================


df_silver = (
    df_silver
    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )
)


print(
    "Metadado Silver adicionado com sucesso!"
)

Metadado Silver adicionado com sucesso!


In [0]:
# ==================================================
# GRAVAÇÃO DA CAMADA SILVER
# ECOMMERCE PEDIDOS
# ==================================================


(
    df_silver
    .write
    .options(**adls_options)
    .format("delta")
    .mode("append")
    .save(
        caminho_silver_pedidos
    )
)



print(
    "Silver Delta gravada com sucesso!"
)


print(
    "Destino:"
)


print(
    caminho_silver_pedidos
)

Silver Delta gravada com sucesso!
Destino:
abfss://squad2@internshipdatalake.dfs.core.windows.net/grupo3/silver/ecommerce_pedidos


In [0]:
# ==================================================
# VALIDAÇÃO FINAL DA CAMADA SILVER
# ECOMMERCE PEDIDOS
# ==================================================


df_silver_validacao = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(
        caminho_silver_pedidos
    )
)



print("====================================")
print("VALIDAÇÃO SILVER PEDIDOS")
print("====================================")


print(
    "Quantidade de registros Silver:",
    df_silver_validacao.count()
)


print(
    "\nSchema Silver:"
)


df_silver_validacao.printSchema()

VALIDAÇÃO SILVER PEDIDOS
Quantidade de registros Silver: 766

Schema Silver:
root
 |-- id_pedido: long (nullable = true)
 |-- id_cliente: long (nullable = true)
 |-- id_endereco_entrega: long (nullable = true)
 |-- dt_pedido: timestamp (nullable = true)
 |-- status_pedido: string (nullable = true)
 |-- valor_total: double (nullable = true)
 |-- valor_frete: double (nullable = true)
 |-- metodo_pagamento: string (nullable = true)
 |-- dt_previsao_entrega: timestamp (nullable = true)
 |-- dt_ultima_atualizacao_status: timestamp (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- bronze_source_file: string (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- dia: integer (nullable = true)
 |-- hora: integer (nullable = true)
 |-- silver_processed_at: timestamp (nullable = true)



# ✅ Conclusão

A camada Silver da tabela `ecommerce_pedidos` foi implementada com sucesso utilizando Delta Lake no Azure Data Lake Gen2.

Nesta etapa, os dados provenientes da camada Bronze foram processados e preparados para consumo analítico, aplicando regras de qualidade e padronização.

Durante o processamento foram realizadas as seguintes etapas:

- Leitura dos dados da camada Bronze utilizando Apache Spark;
- Remoção de registros duplicados utilizando `id_pedido` como chave de negócio;
- Validação de campos obrigatórios;
- Validação de consistência dos valores financeiros;
- Padronização de campos textuais;
- Inclusão do metadado de processamento:
  - `silver_processed_at`;
- Gravação dos dados tratados no formato Delta Lake.

A tabela foi armazenada em:

`grupo3/silver/ecommerce_pedidos`

A validação final confirmou:

- 766 registros tratados na camada Silver;
- Redução de inconsistências identificadas durante o processo de qualidade;
- Estrutura Delta criada corretamente (`_delta_log`);
- Dados preparados para consumo das próximas camadas analíticas.

A camada Silver representa uma versão tratada e confiável dos dados, servindo como base para a construção da camada Gold, onde serão criados indicadores e métricas de negócio.